# Solving a Real Problem Using Our Model: The Discrete Flashing Ratchet (DFR)
The following is adapted from {cite}`fim_mjp` and shows a step by step example for solving problems using our [trained model from Hugging Face](https://huggingface.co/FIM4Science/fim-mjp). This notebook is a detailed example of solving a physics problem using our model using the following approach:

1. Understanding the problem
2. Loading and understanding the data
3. Using our model to infer the transition rates
4. Using the inferred transition rates to solve the original problem 




In statistical physics, the ratchet effect refers to the rectification of thermal fluctuations into directed motion to produce work, and goes all the way back to Feynman {cite}`feynman1965feynman`.

Here we consider a simple example thereof, in which a Brownian particle, immersed in a thermal bath at unit temperature, moves on a one-dimensional lattice.
The particle is subject to a linear, periodic and asymmetric potential of maximum height $2V$ that is switched on and off at a constant rate $r$.
The potential has three possible values when is switched on, which correspond to three of the states of the system. 
The particle jumps among them with rate $f_{ij}^{\tiny{\text{ on}}}$.

When the potential is switched off, the particle jumps freely with rate $f_{ij}^{\tiny{\text{ off}}}$.
We can therefore think of the system as a six-state system, as illustrated here: 
![](dfr_schema.png)

Similar to {cite}`roldan10`, we now define the transition rates as
```{math}
    f_{ij}^{\tiny{\text{on}}} = \exp \left( -\frac{V}{2}(j-i)\right), \, \, \, \text{for} \, \, i, j \in (0, 1, 2); \quad f_{ij}^{\tiny{\text{ off}}} = b, \, \, \, \text{for} \, \, i, j \in (3, 4, 5).
```
Given these specifics, we consider the parameter set $(V, r, B) = (1, 1, 1)$  
together with the dataset simulated by {cite}`neural_mjp`,

So given our data, we want to recover the theoretical transition rates, as a q-matrix.[^footnote1]
[^footnote1]: That is with the diagonal entries normalized to the negative sum of the transition rates

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
np.set_printoptions(linewidth=100)
from transformers.utils import logging
logging.disable_progress_bar()
from datasets.utils.logging import disable_progress_bar 
disable_progress_bar()

In [ ]:
rates_on = np.array([[np.exp(-0.5 * (j - i)) - 1 * (i == j) for j in range(3)] for i in range(3)])
rates_off = np.ones(shape=(3, 3))
np.fill_diagonal(rates_off, 0)
id3 = np.eye(3)
q_matrix = np.zeros(shape=(6, 6))
q_matrix[:3, :3] = rates_on
q_matrix[3:, 3:] = rates_off
q_matrix[3:, :3] = id3
q_matrix[:3, 3:] = id3
diagonal = -np.sum(q_matrix, axis=1)
np.fill_diagonal(q_matrix, diagonal)
q_matrix

## Loading and Exploring the Data
We start by loading the simulated data from Hugging Face:

In [ ]:
# Loading the Discrete Flashing Ratchet (DFR) dataset from Huggingface
import torch
from datasets import load_dataset

data = load_dataset("FIM4Science/mjp", download_mode="force_redownload", name="default")
data.set_format("torch")

The model will later use the 'observation_grid', 'observation_values', 'seq_lengths' features to estimate the transition rates.

Here the observation grid is constant over all paths, consisting of 100 evenly spaced points on $[0,1]$. The sequence lengths of the dataset is therefore just 100 for all paths.

In [ ]:
data["train"]["observation_grid"][0][0, 0, :, 0]
print(data["train"]["observation_values"][0][0, 0, :, 0])
print(data["train"]["observation_values"][0][0, 300, :, 0])

The observation values contain the state of the processes for all paths and all time points. 
```{warning} In practice these labels will rarely be directly observed values, since those will, in most cases, have to be computed as part of the preprocessing.
```
We can visualize the first 3 paths to get a feeling for the processes:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

cmap = mcolors.LinearSegmentedColormap.from_list("custom_cmap", ["#D55E00", "#56B4E9"], N=3)
colors = [mcolors.to_hex(cmap(i)) for i in range(3)]

ts = data["train"]["observation_grid"][0][0, 0, :, 0]
three_paths = data["train"]["observation_values"][0][0, :3, :, 0]
for i in range(3):
    plt.plot(ts, three_paths[i], "x--", c=colors[i])
plt.show()

## Inferring Transition Rates

To infer the $q$-matrix we first load our trained model:

In [ ]:
from fim.trainers.utils import get_accel_type
from fim.models.mjp import FIMMJP

fimmjp = FIMMJP.from_pretrained("FIM4Science/fim-mjp", trust_remote_code=True)

device = get_accel_type()
fimmjp = fimmjp.to(device)
fimmjp.eval()

As noted in {cite}`fim_mjp`[^footnote2] it suffices to look at a small context window of 300 paths with 50 observation values each. We therefore infer the 
transition rates batchwise to demonstrate how little data might be needed. 

[^footnote2]: Section 4.1 and E.4

In [ ]:
# copy data to device
batch = {
    k: v.to(device)[0]
    for k, v in data["train"][:1].items()
    if k not in ["intensity_matrices", "adjacency_matrices", "initial_distributions"]
}  # data without any information, we seek to find

n_paths = 300
total_n_paths = batch["observation_grid"][0].shape[0]
statistics = 50

outputs = []

with torch.no_grad():
    for _ in range(statistics):
        paths_idx = torch.randperm(total_n_paths)[:n_paths]
        mini_batch = batch.copy()
        mini_batch["observation_grid"] = batch["observation_grid"][:, paths_idx]
        mini_batch["observation_values"] = batch["observation_values"][:, paths_idx]
        mini_batch["seq_lengths"] = batch["seq_lengths"][:, paths_idx]
        
        outputs.append(
            fimmjp(mini_batch, n_states=6)["intensity_matrices"]
        )  # We are currently not interested in variances or initial conditions

This results in the following inferred transition rates:

In [ ]:
mean_rates = torch.mean(torch.stack(outputs, dim=0), dim=0)[0].to("cpu").numpy()
print(mean_rates)
plt.matshow(mean_rates)
plt.colorbar()
plt.show()

and the following variances:

In [ ]:
variances = torch.var(torch.stack(outputs, dim=0), dim=0)[0].to("cpu").numpy()
print(variances)
plt.matshow(variances)
plt.colorbar()
plt.show()

Using the previously computed theoretical transition rates we can look at the pointwise error:

In [ ]:
error = mean_rates - q_matrix
print(error)
plt.matshow(error)
plt.colorbar()
plt.show()

## Inferring the Parameters
We will now use the transition rates to recover the parameters used to generate the dataset. Recall the formula we previously used to calculate the theoretical transition rates:
```{math}
    f_{ij}^{\tiny{\text{on}}} = \exp \left( -\frac{V}{2}(j-i)\right), \, \, \, \text{for} \, \, i, j \in (0, 1, 2); \quad f_{ij}^{\tiny{\text{ off}}} = b, \, \, \, \text{for} \, \, i, j \in (3, 4, 5).
```
The $q$-matrix (transition rate matrix) is therefore given by 
```{math}
q=\begin{pmatrix}
f_{ij}^{\tiny{\text{on}}} & r\cdot I_3\\
r\cdot I_3 & f_{ij}^{\tiny{\text{ off}}}
\end{pmatrix}
```
since the transition rates between $(i,j,\text{on})$ and $(i,j,\text{off})$ (and vice versa) are given by the parameter $r$.

This system is evidently overdetermined. For simplicity we assume normally distributed independent errors, which results in the following MLEs for each of the parameters:

We can easily recover $b$ using the diagonal:
```{math}
    \hat{b}=\frac{1}{6}\sum_{i=3}^5\sum_{j=3}^51_{i\neq j}f_{ij}^{\tiny{\text{off}}}
```

In [ ]:
lower_matrix = mean_rates[3:, 3:]
np.fill_diagonal(lower_matrix, 0)

B_hat = lower_matrix.sum() / 6
B_hat

Similarly $r$ describes the transition rates from a state $(i,j,\text{on})$ to $(i,j,\text{off})$ and vice versa:
```{math}
    \hat{r}=\frac{1}{6}\sum_{i=0}^2 q_{i,i+3}+q_{i+3,i}
```

In [ ]:
r_hat = sum([mean_rates[i, i + 3] + mean_rates[i + 3, i] for i in range(3)]) / 6
r_hat

And  $V$ can be recovered using $(q_{i,j})_{i,j=1}^3$:
```{math}
    \hat{V}=\frac{1}{6}\left(-2\cdot(\ln q_{0,1}+\ln q_{1,2})+2\cdot(\ln q_{1,0}+\ln q_{2,1})- \ln q_{0,2}+ \ln q_{2,0}\right)
```

In [ ]:
V_hat = (
    -2 * (np.log(mean_rates[0, 1]) + np.log(mean_rates[1, 2]))
    + 2 * (np.log(mean_rates[1, 0]) + np.log(mean_rates[2, 1]))
    - np.log(mean_rates[0, 2])
    + np.log(mean_rates[2, 0])
) / 6
V_hat

Which results in the following inferred parameters:

In [ ]:
(V_hat, r_hat, B_hat)